## 0. Khởi tạo môi trường


In [28]:
import os
import sys
import re
import time
import math
import sqlite3
import shutil
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import orjson
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score, f1_score, mean_absolute_error
from sklearn.model_selection import StratifiedShuffleSplit

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'sans-serif']
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

for folder in ['data/chart_img', 'data/df', 'data/feature', 'data/dataset']:
    Path(folder).mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]} | PyArrow: {pa.__version__} | NumPy: {np.__version__} | Pandas: {pd.__version__}")
print("Khởi tạo môi trường thành công!")


Python: 3.11.16 | PyArrow: 17.0.0 | NumPy: 1.26.4 | Pandas: 3.0.6
Khởi tạo môi trường thành công!


## 1. Import dataset


In [29]:
json_candidates = [
    Path("data/dataset/TripAdvisor_EN.json"),
    Path("/kaggle/input/dap_dts/TripAdvisor_EN.json"),
    Path("/kaggle/input/dap-dts/TripAdvisor_EN.json"),
    Path("TripAdvisor_EN.json")
]
dataset_path = next((p for p in json_candidates if p.exists()), None)
if dataset_path is None and Path("/kaggle/input").exists():
    found_jsons = list(Path("/kaggle/input").rglob("TripAdvisor_EN.json"))
    if found_jsons:
        dataset_path = found_jsons[0]
if dataset_path is None:
    dataset_path = Path("data/dataset/TripAdvisor_EN.json")

t0 = time.time()
with open(dataset_path, "rb") as f:
    raw_json = orjson.loads(f.read())
load_duration = time.time() - t0
print(f"Đã đọc {len(raw_json):,} bản ghi bằng orjson trong: {load_duration:.4f} giây.")

reviews_list = []
annotations_list = []

for idx, item in enumerate(raw_json, 1):
    review_id = idx
    raw_item_id = item.get("id")
    data_block = item.get("data", {})
    raw_text = unicodedata.normalize("NFC", data_block.get("text", "").strip())
    meta = data_block.get("meta_info", {})

    score_val = float(meta.get("score", 0.0))
    rating_5 = int(round(score_val)) if score_val > 0 else 5
    rating_2 = 1 if rating_5 >= 4 else 0

    reviews_list.append({
        "review_id": review_id,
        "task_id": raw_item_id,
        "ref_id": meta.get("ref_id", f"TripAdvisor_En_{review_id}"),
        "hotel_name": meta.get("hotel name", "").strip(),
        "hotel_address": meta.get("hotel address", "").strip(),
        "location": meta.get("location", "").strip(),
        "star": meta.get("star", "").strip(),
        "score": score_val,
        "rating_5": rating_5,
        "rating_2": rating_2,
        "date": meta.get("date", "").strip(),
        "year": int(meta.get("year", 0)) if str(meta.get("year", "")).isdigit() else None,
        "month": int(meta.get("month", 0)) if str(meta.get("month", "")).isdigit() else None,
        "review_text": raw_text,
        "char_length": len(raw_text),
        "word_count": len(raw_text.split())
    })

    ann_container = item.get("annotations", [])
    if ann_container:
        results = ann_container[0].get("result", [])
        entities_dict = {}
        sentiments_dict = {}

        for res in results:
            res_id = res.get("id")
            fn = res.get("from_name")
            val = res.get("value", {})
            lbls = val.get("labels", [])
            if not lbls:
                continue
            if fn == "entities":
                entities_dict[res_id] = (lbls[0], int(val.get("start", 0)), int(val.get("end", 0)), val.get("text", ""))
            elif fn == "entity_sentiment":
                sentiments_dict[res_id] = lbls[0]

        for res_id, (aspect, start_c, end_c, span_txt) in entities_dict.items():
            sentiment = sentiments_dict.get(res_id, "Neutral")
            annotations_list.append({
                "annotation_id": f"{review_id}_{res_id}",
                "review_id": review_id,
                "aspect": aspect,
                "sentiment": sentiment,
                "start_char": start_c,
                "end_char": end_c,
                "span_text": span_txt.strip()
            })

df_reviews = pd.DataFrame(reviews_list)
df_annotations = pd.DataFrame(annotations_list)

print(f"Tổng số reviews: {len(df_reviews):,} | Tổng số aspect annotations: {len(df_annotations):,}")


Đã đọc 9,990 bản ghi bằng orjson trong: 1.6966 giây.
Tổng số reviews: 9,990 | Tổng số aspect annotations: 54,326


## 2. Dataframe và SQL Queries


### 2.1 Khởi tạo Dataframe


In [30]:
os.makedirs("data/df", exist_ok=True)
db_path = "data/df/chatbot.db"

if Path("chatbot.db").exists():
    Path("chatbot.db").unlink()

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS reviews (
    review_id INTEGER PRIMARY KEY,
    task_id INTEGER,
    ref_id TEXT,
    hotel_name TEXT,
    hotel_address TEXT,
    location TEXT,
    star TEXT,
    score REAL,
    rating_5 INTEGER,
    rating_2 INTEGER,
    date TEXT,
    year INTEGER,
    month INTEGER,
    review_text TEXT,
    char_length INTEGER,
    word_count INTEGER
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS aspect_annotations (
    annotation_id TEXT PRIMARY KEY,
    review_id INTEGER,
    aspect TEXT,
    sentiment TEXT,
    start_char INTEGER,
    end_char INTEGER,
    span_text TEXT,
    FOREIGN KEY (review_id) REFERENCES reviews(review_id)
);
""")

cursor.execute("CREATE INDEX IF NOT EXISTS idx_rev_rating5 ON reviews(rating_5);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_rev_location ON reviews(location);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ann_aspect_sentiment ON aspect_annotations(aspect, sentiment);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ann_review_id ON aspect_annotations(review_id);")

df_reviews.to_sql("reviews", conn, if_exists="replace", index=False)
df_annotations.to_sql("aspect_annotations", conn, if_exists="replace", index=False)
conn.commit()

r_cnt = cursor.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]
a_cnt = cursor.execute("SELECT COUNT(*) FROM aspect_annotations").fetchone()[0]
print(f"Khởi tạo CSDL tại '{db_path}' thành công! Số reviews: {r_cnt:,} | Số annotations: {a_cnt:,}")


Khởi tạo CSDL tại 'data/df/chatbot.db' thành công! Số reviews: 9,990 | Số annotations: 54,326


### 2.2 SQL Queries cho các Research Questions (RQs)


In [31]:
query_1 = """
SELECT
    rating_5 AS rating_tier,
    COUNT(review_id) AS total_reviews,
    ROUND(100.0 * COUNT(review_id) / (SELECT COUNT(*) FROM reviews), 2) AS pct_share,
    CASE
        WHEN rating_5 >= 4 THEN 'Positive (2-Class)'
        ELSE 'Negative/Neutral (2-Class)'
    END AS binary_class
FROM reviews
GROUP BY rating_5
ORDER BY rating_5 ASC;
"""
df_q1 = pd.read_sql_query(query_1, conn)
print("Kết quả Query 1 - RQ1:")
display(df_q1)


Kết quả Query 1 - RQ1:


,rating_tier,total_reviews,pct_share,binary_class
0,1,228,2.28,Negative/Neutral (2-Class)
1,2,237,2.37,Negative/Neutral (2-Class)
2,3,601,6.02,Negative/Neutral (2-Class)
3,4,1776,17.78,Positive (2-Class)
4,5,7148,71.55,Positive (2-Class)


In [32]:
query_2 = """
SELECT
    r.rating_5 AS rating_tier,
    COUNT(a.annotation_id) AS total_rationales,
    ROUND(AVG(a.end_char - a.start_char), 2) AS avg_rationale_chars,
    ROUND(AVG(CAST(a.end_char - a.start_char AS REAL) / r.char_length) * 100, 2) AS avg_evidence_coverage_pct
FROM reviews r
JOIN aspect_annotations a ON r.review_id = a.review_id
WHERE r.char_length > 0
GROUP BY r.rating_5
ORDER BY r.rating_5 ASC;
"""
df_q2 = pd.read_sql_query(query_2, conn)
print("Kết quả Query 2 - RQ1:")
display(df_q2)


Kết quả Query 2 - RQ1:


,rating_tier,total_rationales,avg_rationale_chars,avg_evidence_coverage_pct
0,1,1088,74.16,14.42
1,2,1180,72.66,14.02
2,3,3222,67.02,13.70
3,4,9821,63.42,14.12
4,5,39015,62.77,14.69


In [33]:
query_3 = """
SELECT
    a1.aspect AS aspect_x,
    a2.aspect AS aspect_y,
    COUNT(DISTINCT a1.review_id) AS co_occurrence_count
FROM aspect_annotations a1
JOIN aspect_annotations a2 ON a1.review_id = a2.review_id AND a1.aspect < a2.aspect
GROUP BY aspect_x, aspect_y
ORDER BY co_occurrence_count DESC;
"""
df_q3 = pd.read_sql_query(query_3, conn)
print("Kết quả Query 3 - RQ2:")
display(df_q3)


Kết quả Query 3 - RQ2:


,aspect_x,aspect_y,co_occurrence_count
0,Facility,Service,6932
1,Experience,Service,5512
2,Amenity,Service,5224
3,Amenity,Facility,4496
4,Experience,Facility,4483
5,Loyalty,Service,3717
6,Amenity,Experience,3364
7,Facility,Loyalty,2873
8,Experience,Loyalty,2414
9,Amenity,Loyalty,2181


In [34]:
query_4 = """
SELECT
    a1.aspect AS aspect_pos,
    a2.aspect AS aspect_neg,
    COUNT(DISTINCT a1.review_id) AS conflicting_reviews_count
FROM aspect_annotations a1
JOIN aspect_annotations a2 ON a1.review_id = a2.review_id AND a1.aspect != a2.aspect
WHERE a1.sentiment = 'Positive' AND a2.sentiment = 'Negative'
GROUP BY aspect_pos, aspect_neg
ORDER BY conflicting_reviews_count DESC
LIMIT 10;
"""
df_q4 = pd.read_sql_query(query_4, conn)
print("Kết quả Query 4 - RQ2:")
display(df_q4)


Kết quả Query 4 - RQ2:


,aspect_pos,aspect_neg,conflicting_reviews_count
0,Service,Facility,640
1,Amenity,Facility,480
2,Facility,Service,384
3,Service,Experience,359
4,Experience,Facility,337
5,Facility,Experience,316
6,Amenity,Service,281
7,Amenity,Experience,246
8,Experience,Service,203
9,Service,Amenity,187


In [35]:
query_5 = """
SELECT
    a1.aspect AS aspect_1,
    a2.aspect AS aspect_2,
    COUNT(*) AS collision_count
FROM aspect_annotations a1
JOIN aspect_annotations a2 ON a1.review_id = a2.review_id AND a1.annotation_id < a2.annotation_id
WHERE a1.aspect != a2.aspect
  AND NOT (a1.end_char <= a2.start_char OR a1.start_char >= a2.end_char)
GROUP BY aspect_1, aspect_2
ORDER BY collision_count DESC;
"""
df_q5 = pd.read_sql_query(query_5, conn)
print("Kết quả Query 5 - RQ3:")
display(df_q5)


Kết quả Query 5 - RQ3:


,aspect_1,aspect_2,collision_count
0,Facility,Service,2
1,Amenity,Facility,1
2,Experience,Amenity,1
3,Facility,Experience,1
4,Loyalty,Amenity,1
5,Service,Experience,1
6,Service,Loyalty,1


In [36]:
query_6 = """
SELECT
    r.rating_5,
    a.aspect,
    COUNT(DISTINCT r.review_id) AS review_count,
    SUM(CASE WHEN a.sentiment = 'Positive' THEN 1 ELSE 0 END) AS pos_cnt,
    SUM(CASE WHEN a.sentiment = 'Negative' THEN 1 ELSE 0 END) AS neg_cnt,
    ROUND(CAST(COUNT(r.review_id) AS REAL) / (SELECT COUNT(*) FROM reviews), 4) AS class_prior
FROM reviews r
JOIN aspect_annotations a ON r.review_id = a.review_id
WHERE a.aspect IN ('Service', 'Facility', 'Experience')
GROUP BY r.rating_5, a.aspect
ORDER BY a.aspect, r.rating_5 ASC;
"""
df_q6 = pd.read_sql_query(query_6, conn)
print("Kết quả Query 6 - RQ3:")
display(df_q6)


Kết quả Query 6 - RQ3:


,rating_5,aspect,review_count,pos_cnt,neg_cnt,class_prior
0,1,Experience,140,10,199,0.0215
1,2,Experience,119,30,132,0.0181
2,3,Experience,332,215,220,0.0524
3,4,Experience,1022,1158,220,0.1534
4,5,Experience,4416,6318,101,0.6669
5,1,Facility,147,26,276,0.0321
6,2,Facility,181,76,248,0.0376
7,3,Facility,515,427,493,0.1105
8,4,Facility,1475,2016,415,0.2861
9,5,Facility,5141,7343,196,0.7990


## 3. Data Cleaning và EDA


### 3.1 Kiểm định tính toàn vẹn dữ liệu và kiểm tra Missing Values


In [37]:
missing_reviews = df_reviews.isnull().sum()
missing_ann = df_annotations.isnull().sum()
print("Missing values trong df_reviews:\n", missing_reviews[missing_reviews > 0])
print("Missing values trong df_annotations:\n", missing_ann[missing_ann > 0])

invalid_spans = df_annotations[
    (df_annotations['start_char'] < 0) |
    (df_annotations['start_char'] >= df_annotations['end_char'])
]
print(f"Số lượng annotation spans có tọa độ không hợp lệ: {len(invalid_spans)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

completeness = (1 - df_reviews[['hotel_name', 'location', 'star', 'score', 'date', 'review_text']].isnull().mean()) * 100
sns.barplot(x=completeness.values, y=completeness.index, ax=axes[0], color='#2b5c8f')
axes[0].set_title("Data Completeness Rate (%)", fontweight='bold')
axes[0].set_xlim(0, 105)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_width():.1f}%", (p.get_width() + 1, p.get_y() + p.get_height() / 2), va='center')

sns.histplot(df_reviews['word_count'], bins=50, kde=True, ax=axes[1], color='#e67e22')
axes[1].axvline(df_reviews['word_count'].median(), color='red', linestyle='--', label=f"Median: {df_reviews['word_count'].median():.0f} words")
axes[1].set_title("Distribution of Review Word Counts", fontweight='bold')
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
chart_path_31 = "data/chart_img/31_missing_values_and_data_summary.png"
plt.savefig(chart_path_31)
plt.close()
print(f"Đã lưu biểu đồ làm sạch dữ liệu tại: '{chart_path_31}'")


Missing values trong df_reviews:
 Series([], dtype: int64)
Missing values trong df_annotations:
 Series([], dtype: int64)
Số lượng annotation spans có tọa độ không hợp lệ: 0
Đã lưu biểu đồ làm sạch dữ liệu tại: 'data/chart_img/31_missing_values_and_data_summary.png'


### 3.2 Thống kê mô tả đặc tính văn bản và phân vị độ dài


In [38]:
desc_stats = df_reviews[['char_length', 'word_count', 'score']].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
print("Bảng thống kê mô tả đặc tính văn bản đánh giá:")
display(desc_stats)

plt.figure(figsize=(10, 5))
sns.kdeplot(df_reviews['char_length'], fill=True, color='#27ae60', alpha=0.4)
plt.title("Kernel Density Estimation of Character Lengths", fontweight='bold')
plt.xlabel("Review Character Length")
plt.ylabel("Density")
chart_path_32 = "data/chart_img/32_review_text_length_distribution.png"
plt.savefig(chart_path_32)
plt.close()
print(f"Đã lưu biểu đồ phân bố độ dài tại: '{chart_path_32}'")


Bảng thống kê mô tả đặc tính văn bản đánh giá:


,count,mean,std,min,5%,25%,50%,75%,95%,max
char_length,9990.0,450.664464,200.472604,177.0,210.45,274.0,398.0,618.0,786.0,847.0
word_count,9990.0,82.145846,37.112671,30.0,37.00,49.0,72.0,114.0,146.0,150.0
score,9990.0,4.539439,0.886036,1.0,3.00,4.0,5.0,5.0,5.0,5.0


Đã lưu biểu đồ phân bố độ dài tại: 'data/chart_img/32_review_text_length_distribution.png'


## 4. Data Visualization


### 4.1 Phân bố số sao tổng quan


In [39]:
plt.figure(figsize=(9, 5))
score_counts = df_reviews['rating_5'].value_counts().sort_index()
bars = plt.bar(score_counts.index, score_counts.values, color=['#e74c3c', '#e67e22', '#f1c40f', '#3498db', '#2ecc71'])
plt.title("4.1 Phân bố điểm đánh giá tổng quan (5-Star Rating Distribution)", fontweight='bold')
plt.xlabel("Xếp hạng sao (1 đến 5 sao)")
plt.ylabel("Số lượng đánh giá")
plt.xticks([1, 2, 3, 4, 5])
for bar in bars:
    yval = bar.get_height()
    pct = (yval / len(df_reviews)) * 100
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 100, f"{yval:,}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=10)
plt.ylim(0, max(score_counts.values) * 1.15)
plt.tight_layout()
chart_path_41 = "data/chart_img/41_overall_rating_distribution.png"
plt.savefig(chart_path_41)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_41}'")


Đã lưu biểu đồ: 'data/chart_img/41_overall_rating_distribution.png'


### 4.2 Top 10 khách sạn có số review cao nhất


In [40]:
plt.figure(figsize=(12, 6))
top_hotels = df_reviews["hotel_name"].value_counts().head(10)
hotel_labels = [unicodedata.normalize("NFC", name).strip().title() for name in top_hotels.index]

hotel_avg_scores = df_reviews.groupby("hotel_name")["score"].mean()

sns.barplot(x=top_hotels.values, y=hotel_labels, palette="Blues_r", hue=hotel_labels, legend=False)
plt.title("4.2 Top 10 Khách sạn có review cao nhất", fontweight="bold", fontsize=13, pad=12)
plt.xlabel("Số lượng bài đánh giá", fontsize=11)
plt.ylabel("Khách sạn", fontsize=11)

for i, (orig_name, v) in enumerate(top_hotels.items()):
    avg_s = hotel_avg_scores.get(orig_name, 0.0)
    pct = (v / len(df_reviews)) * 100
    plt.text(v + 1, i, f"{v:,} ({pct:.1f}%) | Điểm TB: {avg_s:.2f}★", va="center", fontsize=9.5)

plt.xlim(0, max(top_hotels.values) * 1.35)
plt.tight_layout()
chart_path_42 = "data/chart_img/42_top10_hotels_highest_reviews.png"
plt.savefig(chart_path_42, dpi=300)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_42}'")


Đã lưu biểu đồ: 'data/chart_img/42_top10_hotels_highest_reviews.png'


### 4.3 Tần suất xuất hiện của các khía cạnh khách sạn


In [41]:
plt.figure(figsize=(9, 5))
aspect_counts = df_annotations['aspect'].value_counts()
sns.barplot(x=aspect_counts.index, y=aspect_counts.values, palette='viridis', hue=aspect_counts.index, legend=False)
plt.title("4.3 Tần suất xuất hiện của các khía cạnh khách sạn", fontweight='bold')
plt.xlabel("Khía cạnh (Aspect)")
plt.ylabel("Số lượng đoạn bằng chứng (Annotations)")
for i, v in enumerate(aspect_counts.values):
    plt.text(i, v + 300, f"{v:,}", ha='center', fontsize=10, fontweight='bold')
plt.ylim(0, max(aspect_counts.values) * 1.12)
plt.tight_layout()
chart_path_43 = "data/chart_img/43_aspect_frequency_distribution.png"
plt.savefig(chart_path_43)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_43}'")


Đã lưu biểu đồ: 'data/chart_img/43_aspect_frequency_distribution.png'


### 4.4 Phân bố cảm xúc theo từng khía cạnh


In [42]:
plt.figure(figsize=(10, 5))
asp_sent = pd.crosstab(df_annotations['aspect'], df_annotations['sentiment'], normalize='index') * 100
cols = [c for c in ['Positive', 'Neutral', 'Negative'] if c in asp_sent.columns]
asp_sent = asp_sent[cols]
colors = {'Positive': '#2ecc71', 'Neutral': '#f1c40f', 'Negative': '#e74c3c'}
ax = asp_sent.plot(kind='bar', stacked=True, figsize=(10, 5), color=[colors[c] for c in cols], edgecolor='none')
plt.title("4.4 Tỷ lệ cảm xúc theo từng khía cạnh khách sạn (%)", fontweight='bold')
plt.xlabel("Khía cạnh (Aspect)")
plt.ylabel("Tỷ lệ phần trăm (%)")
plt.legend(title="Cảm xúc", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=15)
plt.ylim(0, 100)
plt.tight_layout()
chart_path_44 = "data/chart_img/44_aspect_sentiment_breakdown.png"
plt.savefig(chart_path_44)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_44}'")


Đã lưu biểu đồ: 'data/chart_img/44_aspect_sentiment_breakdown.png'


<Figure size 3000x1500 with 0 Axes>

### 4.5 Độ dài đoạn bằng chứng theo khía cạnh


In [43]:
df_annotations['span_len'] = df_annotations['end_char'] - df_annotations['start_char']
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_annotations, x='aspect', y='span_len', palette='Set2', hue='aspect', legend=False, showfliers=False)
plt.title("4.5 Phân bố độ dài ký tự của đoạn bằng chứng (Rationale Span Length)", fontweight='bold')
plt.xlabel("Khía cạnh (Aspect)")
plt.ylabel("Độ dài đoạn văn (Ký tự)")
plt.tight_layout()
chart_path_45 = "data/chart_img/45_rationale_length_by_aspect.png"
plt.savefig(chart_path_45)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_45}'")


Đã lưu biểu đồ: 'data/chart_img/45_rationale_length_by_aspect.png'


### 4.6 Ma trận tương quan đồng xuất hiện giữa các khía cạnh


In [44]:
aspect_list = sorted(df_annotations['aspect'].unique())
co_mat = pd.DataFrame(0, index=aspect_list, columns=aspect_list)

rev_aspects = df_annotations.groupby('review_id')['aspect'].unique()
for asps in rev_aspects:
    for a1 in asps:
        for a2 in asps:
            co_mat.loc[a1, a2] += 1

plt.figure(figsize=(8, 6))
sns.heatmap(co_mat, annot=True, fmt='d', cmap='YlGnBu', cbar=True)
plt.title("4.6 Ma trận đồng xuất hiện giữa các khía cạnh trong cùng review", fontweight='bold')
plt.xlabel("Khía cạnh")
plt.ylabel("Khía cạnh")
plt.tight_layout()
chart_path_46 = "data/chart_img/46_aspect_cooccurrence_matrix.png"
plt.savefig(chart_path_46)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_46}'")


Đã lưu biểu đồ: 'data/chart_img/46_aspect_cooccurrence_matrix.png'


### 4.7 Tỷ lệ phân kỳ cảm xúc giữa các cặp khía cạnh trong cùng review


In [45]:
conflict_mat = pd.DataFrame(0, index=aspect_list, columns=aspect_list)
rev_asp_sent = df_annotations.groupby(['review_id', 'aspect'])['sentiment'].unique()

rev_to_asp = defaultdict(dict)
for (rid, asp), sents in rev_asp_sent.items():
    rev_to_asp[rid][asp] = sents

for rid, asps_dict in rev_to_asp.items():
    for a1, s1 in asps_dict.items():
        for a2, s2 in asps_dict.items():
            if a1 != a2 and 'Positive' in s1 and 'Negative' in s2:
                conflict_mat.loc[a1, a2] += 1

plt.figure(figsize=(8, 6))
sns.heatmap(conflict_mat, annot=True, fmt='d', cmap='Reds', cbar=True)
plt.title("4.7 Ma trận phân kỳ cảm xúc (Aspect A Pos vs Aspect B Neg)", fontweight='bold')
plt.xlabel("Khía cạnh tiêu cực (Negative)")
plt.ylabel("Khía cạnh tích cực (Positive)")
plt.tight_layout()
chart_path_47 = "data/chart_img/47_sentiment_divergence_rate.png"
plt.savefig(chart_path_47)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_47}'")


Đã lưu biểu đồ: 'data/chart_img/47_sentiment_divergence_rate.png'


### 4.8 Phân bố số lượng từ của bài đánh giá theo mức xếp hạng sao


In [46]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df_reviews, x='rating_5', y='word_count', palette='Blues', hue='rating_5', legend=False, showfliers=False)
plt.title("4.8 Phân bố độ dài từ (Word Count) theo thang điểm 5 mức", fontweight='bold')
plt.xlabel("Xếp hạng sao (1 đến 5 sao)")
plt.ylabel("Số từ trong bài đánh giá")
plt.tight_layout()
chart_path_48 = "data/chart_img/48_review_wordcount_by_star.png"
plt.savefig(chart_path_48)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_48}'")


Đã lưu biểu đồ: 'data/chart_img/48_review_wordcount_by_star.png'


### 4.9 Mức độ chồng lấn tọa độ văn bản giữa các khía cạnh


In [47]:
collision_records = []
ann_grouped = df_annotations.groupby('review_id')

for rid, group in ann_grouped:
    records = group[['annotation_id', 'aspect', 'start_char', 'end_char']].to_dict('records')
    n = len(records)
    for i in range(n):
        for j in range(i + 1, n):
            r1, r2 = records[i], records[j]
            if r1['aspect'] != r2['aspect']:
                if not (r1['end_char'] <= r2['start_char'] or r1['start_char'] >= r2['end_char']):
                    pair_label = f"{min(r1['aspect'], r2['aspect'])} & {max(r1['aspect'], r2['aspect'])}"
                    collision_records.append(pair_label)

plt.figure(figsize=(8, 4.5))
if collision_records:
    col_counts = pd.Series(collision_records).value_counts()
    sns.barplot(x=col_counts.values, y=col_counts.index, color='#c0392b')
    plt.title("4.9 Số trường hợp chồng lấn tọa độ ký tự giữa các khía cạnh", fontweight='bold')
    plt.xlabel("Số vụ va chạm tọa độ văn bản")
    for i, v in enumerate(col_counts.values):
        plt.text(v + 0.1, i, str(v), va='center', fontweight='bold')
    plt.xlim(0, max(col_counts.values) + 1)
else:
    plt.text(0.5, 0.5, "Không phát hiện va chạm tọa độ", ha='center', va='center')
    plt.title("4.9 Số trường hợp chồng lấn tọa độ giữa các khía cạnh", fontweight='bold')
plt.tight_layout()
chart_path_49 = "data/chart_img/49_cross_aspect_span_overlap.png"
plt.savefig(chart_path_49)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_49}'")


Đã lưu biểu đồ: 'data/chart_img/49_cross_aspect_span_overlap.png'


### 4.10 Phân bố số lượng khía cạnh được đề cập trên mỗi bài đánh giá


In [48]:
aspects_per_rev = df_annotations.groupby('review_id')['aspect'].nunique()
full_aspects_per_rev = df_reviews['review_id'].map(aspects_per_rev).fillna(0).astype(int)

plt.figure(figsize=(9, 5))
counts_series = full_aspects_per_rev.value_counts().sort_index()
sns.barplot(x=counts_series.index, y=counts_series.values, palette='mako', hue=counts_series.index, legend=False)
plt.title("4.10 Phân bố số lượng khía cạnh được đề cập trên mỗi bài đánh giá", fontweight='bold')
plt.xlabel("Số khía cạnh phân biệt (Aspect Count)")
plt.ylabel("Số lượng bài đánh giá")
for i, v in enumerate(counts_series.values):
    pct = (v / len(df_reviews)) * 100
    plt.text(i, v + 80, f"{v:,}\n({pct:.1f}%)", ha='center', fontsize=9)
plt.ylim(0, max(counts_series.values) * 1.15)
plt.tight_layout()
chart_path_410 = "data/chart_img/410_aspects_per_review_distribution.png"
plt.savefig(chart_path_410)
plt.close()
print(f"Đã lưu biểu đồ: '{chart_path_410}'")


Đã lưu biểu đồ: 'data/chart_img/410_aspects_per_review_distribution.png'


### 4.11 Sơ đồ Quy trình Nghiên cứu (Research Pipeline)


In [49]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, Polygon, Arc

os.makedirs("data/chart_img", exist_ok=True)
os.makedirs("docs/main paper/chart_img", exist_ok=True)

plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.family"] = "sans-serif"

def draw_rounded_card(ax, x, y, w, h, bg="#FFFFFF", edgecolor="none", lw=1.0, radius=0.24, shadow=True, zorder=3):
    if shadow:
        sh = FancyBboxPatch(
            (x + 0.03, y - 0.05), w, h,
            boxstyle=f"round,pad=0.0,rounding_size={radius}",
            facecolor="#000000", alpha=0.055, edgecolor="none", zorder=zorder - 1
        )
        ax.add_patch(sh)
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.0,rounding_size={radius}",
        facecolor=bg, edgecolor=edgecolor, linewidth=lw, zorder=zorder
    )
    ax.add_patch(box)

def draw_icon_data_layers(ax, cx, cy, scale=1.0, zorder=6):
    color = "#2997FF"
    w = 0.34 * scale
    h = 0.17 * scale
    for idx, dy in enumerate([-0.14 * scale, 0.0, 0.14 * scale]):
        y0 = cy + dy
        pts = np.array([
            [cx, y0 + h],
            [cx + w, y0],
            [cx, y0 - h],
            [cx - w, y0]
        ])
        if idx < 2:
            pts_halo = np.array([
                [cx, y0 + h + 0.045 * scale],
                [cx + w + 0.045 * scale, y0],
                [cx, y0 - h - 0.045 * scale],
                [cx - w - 0.045 * scale, y0]
            ])
            ax.add_patch(Polygon(pts_halo, closed=True, facecolor="#FFFFFF", edgecolor="none", zorder=zorder + idx*2))
        ax.add_patch(Polygon(pts, closed=True, facecolor=color, edgecolor="#FFFFFF", linewidth=1.5, zorder=zorder + idx*2 + 1))

def draw_icon_preprocessing_doc(ax, cx, cy, scale=1.0, zorder=6):
    color = "#8E6CEF"
    dw, dh = 0.44 * scale, 0.56 * scale
    x0, y0 = cx - dw / 2 - 0.03 * scale, cy - dh / 2
    doc = FancyBboxPatch(
        (x0, y0), dw, dh,
        boxstyle=f"round,pad=0.0,rounding_size={0.07*scale}",
        facecolor="none", edgecolor=color, linewidth=2.8, zorder=zorder
    )
    ax.add_patch(doc)
    ax.plot([x0 + 0.10*scale, x0 + dw - 0.10*scale], [y0 + dh - 0.16*scale, y0 + dh - 0.16*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([x0 + 0.10*scale, x0 + dw - 0.16*scale], [y0 + dh - 0.28*scale, y0 + dh - 0.28*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    mcx, mcy = x0 + dw - 0.05*scale, y0 + 0.11*scale
    r = 0.125 * scale
    ax.add_patch(Circle((mcx, mcy), r + 0.04*scale, facecolor="#FFFFFF", edgecolor="none", zorder=zorder+2))
    ax.add_patch(Circle((mcx, mcy), r, facecolor="#FFFFFF", edgecolor=color, linewidth=2.8, zorder=zorder+3))
    ax.plot([mcx + r*0.7, mcx + r*1.55], [mcy - r*0.7, mcy - r*1.55],
            color=color, lw=3.0, solid_capstyle="round", zorder=zorder+3)

def draw_icon_circular_arrow(ax, cx, cy, scale=1.0, zorder=6):
    color = "#F5428D"
    r = 0.27 * scale
    arc = Arc((cx, cy), 2*r, 2*r, angle=0, theta1=75, theta2=395, color=color, lw=3.0, capstyle="round", zorder=zorder)
    ax.add_patch(arc)
    ang = math.radians(68)
    ax_pt = cx + r * math.cos(ang)
    ay_pt = cy + r * math.sin(ang)
    head = np.array([
        [ax_pt + 0.08*scale, ay_pt + 0.01*scale],
        [ax_pt - 0.07*scale, ay_pt + 0.09*scale],
        [ax_pt - 0.04*scale, ay_pt - 0.08*scale]
    ])
    ax.add_patch(Polygon(head, closed=True, facecolor=color, edgecolor=color, lw=1.0, zorder=zorder+1))

def draw_icon_chart_nodes(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF5733"
    w, h = 0.52 * scale, 0.44 * scale
    x0, y0 = cx - w/2, cy - h/2
    ax.plot([x0, x0, x0 + w], [y0 + h, y0, y0], color=color, lw=2.8, solid_capstyle="round", zorder=zorder)
    pts = [
        (x0 + 0.12*scale, y0 + 0.16*scale),
        (x0 + 0.28*scale, y0 + 0.30*scale),
        (x0 + 0.45*scale, y0 + 0.21*scale)
    ]
    xs, ys = zip(*pts)
    ax.plot(xs, ys, color=color, lw=2.6, zorder=zorder+1)
    for px, py in pts:
        ax.add_patch(Circle((px, py), 0.048*scale, facecolor="#FFFFFF", edgecolor=color, lw=2.5, zorder=zorder+2))

def draw_icon_optimization(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF9500"
    bw, bh = 0.54 * scale, 0.11 * scale
    bar = FancyBboxPatch(
        (cx - bw/2, cy - bh/2), bw, bh,
        boxstyle=f"round,pad=0.0,rounding_size={0.055*scale}",
        facecolor="none", edgecolor=color, linewidth=2.6, zorder=zorder
    )
    ax.add_patch(bar)
    ax.plot([cx, cx], [cy + 0.31*scale, cy + 0.11*scale], color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx - 0.09*scale, cx, cx + 0.09*scale], [cy + 0.20*scale, cy + 0.10*scale, cy + 0.20*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx, cx], [cy - 0.31*scale, cy - 0.11*scale], color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx - 0.09*scale, cx, cx + 0.09*scale], [cy - 0.20*scale, cy - 0.10*scale, cy - 0.20*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)

def draw_icon_neural_network(ax, cx, cy, scale=1.0, zorder=6):
    cols_x = [cx - 0.36*scale, cx - 0.12*scale, cx + 0.12*scale, cx + 0.36*scale]
    cols_y = [
        [cy - 0.14*scale, cy + 0.14*scale],
        [cy - 0.26*scale, cy, cy + 0.26*scale],
        [cy - 0.26*scale, cy, cy + 0.26*scale],
        [cy - 0.14*scale, cy + 0.14*scale]
    ]
    col_colors = ["#8CE038", "#4CC764", "#1DB096", "#0099C6"]
    for c_i in range(3):
        c_mid = col_colors[c_i + 1]
        for y1 in cols_y[c_i]:
            for y2 in cols_y[c_i + 1]:
                ax.plot([cols_x[c_i], cols_x[c_i+1]], [y1, y2], color=c_mid, lw=1.7, alpha=0.85, zorder=zorder)
    for c_i in [1, 2]:
        ax.plot([cols_x[c_i], cols_x[c_i]], [cols_y[c_i][0], cols_y[c_i][-1]], color=col_colors[c_i], lw=1.4, alpha=0.6, zorder=zorder)
    for c_i in range(4):
        for y_n in cols_y[c_i]:
            ax.add_patch(Circle((cols_x[c_i], y_n), 0.052*scale, facecolor="#FFFFFF", edgecolor=col_colors[c_i], lw=2.3, zorder=zorder+2))

def build_figure_411():
    fig, ax = plt.subplots(figsize=(16.5, 5.5), dpi=300)
    ax.set_xlim(0, 16.5)
    ax.set_ylim(0, 5.5)
    ax.axis("off")

    draw_rounded_card(ax, 0.10, 0.10, 16.30, 5.30, bg="#F2F2F6", edgecolor="#E2E2E8", lw=1.5, radius=0.32, shadow=False, zorder=1)

    card_w = 2.08
    card_h = 1.92
    card_y = 1.38
    gap = 0.22
    start_x = 0.50

    steps = [
        ("Data", "9,990 Reviews", draw_icon_data_layers),
        ("Preprocessing", "1.pdf & NLTK", draw_icon_preprocessing_doc),
        ("Feature & EBR", "20 Feats • Parquet", draw_icon_circular_arrow),
        ("Multi-Task", "BERT + CORN", draw_icon_chart_nodes),
        ("Optimization", "Overlap & FP16", draw_icon_optimization),
    ]

    for i, (title, subtitle, icon_fn) in enumerate(steps):
        cx_box = start_x + i * (card_w + gap)
        draw_rounded_card(ax, cx_box, card_y, card_w, card_h, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.22, shadow=True, zorder=3)
        center_x = cx_box + card_w / 2
        icon_fn(ax, center_x, card_y + 1.26, scale=0.95, zorder=6)
        ax.text(center_x, card_y + 0.57, title, ha="center", va="center",
                fontsize=14.0, fontweight="bold", color="#1D1D1F", zorder=7)
        ax.text(center_x, card_y + 0.24, subtitle, ha="center", va="center",
                fontsize=11.2, fontweight="normal", color="#515154", zorder=7)

        if i < len(steps) - 1:
            x1 = cx_box + card_w
            x2 = cx_box + card_w + gap
            ym = card_y + card_h / 2
            ax.plot([x1, x2], [ym, ym], color="#86868B", lw=2.0, zorder=2)

    last_card_right = start_x + 4 * (card_w + gap) + card_w
    right_box_x = 12.12
    right_box_y = 1.38
    right_box_w = 3.85
    right_box_h = 3.72
    ym_arrow = card_y + card_h / 2
    ax.plot([last_card_right, right_box_x - 0.04], [ym_arrow, ym_arrow], color="#86868B", lw=2.0, zorder=2)
    ax.plot([right_box_x - 0.16, right_box_x - 0.03, right_box_x - 0.16],
            [ym_arrow + 0.09, ym_arrow, ym_arrow - 0.09], color="#86868B", lw=2.0, solid_capstyle="round", zorder=2)

    draw_rounded_card(ax, right_box_x, right_box_y, right_box_w, right_box_h,
                      bg="#F5F5F7", edgecolor="#FFFFFF", lw=2.2, radius=0.26, shadow=True, zorder=3)
    rb_border = FancyBboxPatch(
        (right_box_x, right_box_y), right_box_w, right_box_h,
        boxstyle="round,pad=0.0,rounding_size=0.26",
        facecolor="none", edgecolor="#D8D8DE", linewidth=1.2, zorder=4
    )
    ax.add_patch(rb_border)

    rcx = right_box_x + right_box_w / 2
    ax.text(rcx, right_box_y + right_box_h - 0.34, "6-Aspect Heads", ha="center", va="center",
            fontsize=15.5, fontweight="bold", color="#1D1D1F", zorder=7)

    aspect_names = ["Service", "Facility", "Exper.", "Amenity", "Loyalty", "Brand."]
    bar_w = 0.44
    bar_h = 1.24
    bar_gap = 0.11
    total_bars_w = 6 * bar_w + 5 * bar_gap
    bar_start_x = rcx - total_bars_w / 2
    bar_y = right_box_y + 1.88
    bar_centers = []

    for a_i, a_lbl in enumerate(aspect_names):
        bx = bar_start_x + a_i * (bar_w + bar_gap)
        bcx = bx + bar_w / 2
        bar_centers.append(bcx)
        pill = FancyBboxPatch(
            (bx, bar_y), bar_w, bar_h,
            boxstyle="round,pad=0.0,rounding_size=0.08",
            facecolor="#0099C6", edgecolor="none", zorder=5
        )
        ax.add_patch(pill)
        for d_i in range(3):
            ax.add_patch(Circle((bcx, bar_y + bar_h - 0.12 - d_i * 0.065), 0.016, facecolor="#FFFFFF", edgecolor="none", zorder=6))
        ax.text(bcx, bar_y + 0.45, a_lbl, ha="center", va="center", rotation=90,
                fontsize=9.2, fontweight="bold", color="#FFFFFF", zorder=7)

    y_tree_top = bar_y
    y_tree_horiz = bar_y - 0.16
    y_tree_bot = bar_y - 0.38
    for bcx in bar_centers:
        ax.plot([bcx, bcx], [y_tree_top, y_tree_horiz], color="#0099C6", lw=1.9, zorder=5)
    ax.plot([bar_centers[0], bar_centers[-1]], [y_tree_horiz, y_tree_horiz], color="#0099C6", lw=1.9, zorder=5)
    ax.plot([rcx, rcx], [y_tree_horiz, y_tree_bot], color="#0099C6", lw=1.9, zorder=5)

    draw_icon_neural_network(ax, rcx, right_box_y + 1.04, scale=0.95, zorder=6)

    ax.text(rcx, right_box_y + 0.45, "CRAM-ABSA Model", ha="center", va="center",
            fontsize=15.5, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(rcx, right_box_y + 0.18, "modelv1.pt • Stars & Review", ha="center", va="center",
            fontsize=11.0, fontweight="normal", color="#515154", zorder=7)

    pill_x = start_x
    pill_w = (right_box_x + right_box_w) - start_x
    pill_y = 0.42
    pill_h = 0.64
    draw_rounded_card(ax, pill_x, pill_y, pill_w, pill_h, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.31, shadow=False, zorder=3)
    pcx = pill_x + pill_w / 2
    pcy = pill_y + pill_h / 2
    split_x = pcx + 0.85
    ax.text(split_x - 0.08, pcy, "Dual-Stream NLTK & EBR Grounding Principles",
            ha="right", va="center", fontsize=14.2, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(split_x + 0.08, pcy, "inform all stages of the pipeline",
            ha="left", va="center", fontsize=14.2, fontweight="normal", color="#1D1D1F", zorder=7)

    for p in [
        "data/chart_img/411_research_pipeline.png",
        "docs/main paper/chart_img/411_research_pipeline.png",
    ]:
        plt.savefig(p, bbox_inches="tight", pad_inches=0.04, facecolor="none")
    plt.close(fig)
    print("Generated refined 411_research_pipeline.png successfully.")

build_figure_411()


Generated refined 411_research_pipeline.png successfully.


### 4.12 Kiến trúc Tổng thể Đề xuất (Abstract Architecture of Proposed Method)


In [50]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, Polygon, Arc

os.makedirs("data/chart_img", exist_ok=True)
os.makedirs("docs/main paper/chart_img", exist_ok=True)

plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.family"] = "sans-serif"

def draw_rounded_card(ax, x, y, w, h, bg="#FFFFFF", edgecolor="none", lw=1.0, radius=0.24, shadow=True, zorder=3):
    if shadow:
        sh = FancyBboxPatch(
            (x + 0.03, y - 0.05), w, h,
            boxstyle=f"round,pad=0.0,rounding_size={radius}",
            facecolor="#000000", alpha=0.055, edgecolor="none", zorder=zorder - 1
        )
        ax.add_patch(sh)
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.0,rounding_size={radius}",
        facecolor=bg, edgecolor=edgecolor, linewidth=lw, zorder=zorder
    )
    ax.add_patch(box)

def draw_icon_data_layers(ax, cx, cy, scale=1.0, zorder=6):
    color = "#2997FF"
    w = 0.34 * scale
    h = 0.17 * scale
    for idx, dy in enumerate([-0.14 * scale, 0.0, 0.14 * scale]):
        y0 = cy + dy
        pts = np.array([
            [cx, y0 + h],
            [cx + w, y0],
            [cx, y0 - h],
            [cx - w, y0]
        ])
        if idx < 2:
            pts_halo = np.array([
                [cx, y0 + h + 0.045 * scale],
                [cx + w + 0.045 * scale, y0],
                [cx, y0 - h - 0.045 * scale],
                [cx - w - 0.045 * scale, y0]
            ])
            ax.add_patch(Polygon(pts_halo, closed=True, facecolor="#FFFFFF", edgecolor="none", zorder=zorder + idx*2))
        ax.add_patch(Polygon(pts, closed=True, facecolor=color, edgecolor="#FFFFFF", linewidth=1.5, zorder=zorder + idx*2 + 1))

def draw_icon_preprocessing_doc(ax, cx, cy, scale=1.0, zorder=6):
    color = "#8E6CEF"
    dw, dh = 0.44 * scale, 0.56 * scale
    x0, y0 = cx - dw / 2 - 0.03 * scale, cy - dh / 2
    doc = FancyBboxPatch(
        (x0, y0), dw, dh,
        boxstyle=f"round,pad=0.0,rounding_size={0.07*scale}",
        facecolor="none", edgecolor=color, linewidth=2.8, zorder=zorder
    )
    ax.add_patch(doc)
    ax.plot([x0 + 0.10*scale, x0 + dw - 0.10*scale], [y0 + dh - 0.16*scale, y0 + dh - 0.16*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([x0 + 0.10*scale, x0 + dw - 0.16*scale], [y0 + dh - 0.28*scale, y0 + dh - 0.28*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    mcx, mcy = x0 + dw - 0.05*scale, y0 + 0.11*scale
    r = 0.125 * scale
    ax.add_patch(Circle((mcx, mcy), r + 0.04*scale, facecolor="#FFFFFF", edgecolor="none", zorder=zorder+2))
    ax.add_patch(Circle((mcx, mcy), r, facecolor="#FFFFFF", edgecolor=color, linewidth=2.8, zorder=zorder+3))
    ax.plot([mcx + r*0.7, mcx + r*1.55], [mcy - r*0.7, mcy - r*1.55],
            color=color, lw=3.0, solid_capstyle="round", zorder=zorder+3)

def draw_icon_circular_arrow(ax, cx, cy, scale=1.0, zorder=6):
    color = "#F5428D"
    r = 0.27 * scale
    arc = Arc((cx, cy), 2*r, 2*r, angle=0, theta1=75, theta2=395, color=color, lw=3.0, capstyle="round", zorder=zorder)
    ax.add_patch(arc)
    ang = math.radians(68)
    ax_pt = cx + r * math.cos(ang)
    ay_pt = cy + r * math.sin(ang)
    head = np.array([
        [ax_pt + 0.08*scale, ay_pt + 0.01*scale],
        [ax_pt - 0.07*scale, ay_pt + 0.09*scale],
        [ax_pt - 0.04*scale, ay_pt - 0.08*scale]
    ])
    ax.add_patch(Polygon(head, closed=True, facecolor=color, edgecolor=color, lw=1.0, zorder=zorder+1))

def draw_icon_chart_nodes(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF5733"
    w, h = 0.52 * scale, 0.44 * scale
    x0, y0 = cx - w/2, cy - h/2
    ax.plot([x0, x0, x0 + w], [y0 + h, y0, y0], color=color, lw=2.8, solid_capstyle="round", zorder=zorder)
    pts = [
        (x0 + 0.12*scale, y0 + 0.16*scale),
        (x0 + 0.28*scale, y0 + 0.30*scale),
        (x0 + 0.45*scale, y0 + 0.21*scale)
    ]
    xs, ys = zip(*pts)
    ax.plot(xs, ys, color=color, lw=2.6, zorder=zorder+1)
    for px, py in pts:
        ax.add_patch(Circle((px, py), 0.048*scale, facecolor="#FFFFFF", edgecolor=color, lw=2.5, zorder=zorder+2))

def draw_icon_optimization(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF9500"
    bw, bh = 0.54 * scale, 0.11 * scale
    bar = FancyBboxPatch(
        (cx - bw/2, cy - bh/2), bw, bh,
        boxstyle=f"round,pad=0.0,rounding_size={0.055*scale}",
        facecolor="none", edgecolor=color, linewidth=2.6, zorder=zorder
    )
    ax.add_patch(bar)
    ax.plot([cx, cx], [cy + 0.31*scale, cy + 0.11*scale], color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx - 0.09*scale, cx, cx + 0.09*scale], [cy + 0.20*scale, cy + 0.10*scale, cy + 0.20*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx, cx], [cy - 0.31*scale, cy - 0.11*scale], color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx - 0.09*scale, cx, cx + 0.09*scale], [cy - 0.20*scale, cy - 0.10*scale, cy - 0.20*scale],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+1)

def draw_icon_neural_network(ax, cx, cy, scale=1.0, zorder=6):
    cols_x = [cx - 0.36*scale, cx - 0.12*scale, cx + 0.12*scale, cx + 0.36*scale]
    cols_y = [
        [cy - 0.14*scale, cy + 0.14*scale],
        [cy - 0.26*scale, cy, cy + 0.26*scale],
        [cy - 0.26*scale, cy, cy + 0.26*scale],
        [cy - 0.14*scale, cy + 0.14*scale]
    ]
    col_colors = ["#8CE038", "#4CC764", "#1DB096", "#0099C6"]
    for c_i in range(3):
        c_mid = col_colors[c_i + 1]
        for y1 in cols_y[c_i]:
            for y2 in cols_y[c_i + 1]:
                ax.plot([cols_x[c_i], cols_x[c_i+1]], [y1, y2], color=c_mid, lw=1.7, alpha=0.85, zorder=zorder)
    for c_i in [1, 2]:
        ax.plot([cols_x[c_i], cols_x[c_i]], [cols_y[c_i][0], cols_y[c_i][-1]], color=col_colors[c_i], lw=1.4, alpha=0.6, zorder=zorder)
    for c_i in range(4):
        for y_n in cols_y[c_i]:
            ax.add_patch(Circle((cols_x[c_i], y_n), 0.052*scale, facecolor="#FFFFFF", edgecolor=col_colors[c_i], lw=2.3, zorder=zorder+2))

def build_figure_412():
    fig, ax = plt.subplots(figsize=(16.5, 6.2), dpi=300)
    ax.set_xlim(0, 16.5)
    ax.set_ylim(0, 6.2)
    ax.axis("off")

    draw_rounded_card(ax, 0.10, 0.10, 16.30, 6.00, bg="#F2F2F6", edgecolor="#E2E2E8", lw=1.5, radius=0.32, shadow=False, zorder=1)

    col1_x = 0.55
    col1_w = 3.35
    card_h_left = 1.92
    y_top_row = 3.68
    y_bot_row = 1.42

    draw_rounded_card(ax, col1_x, y_top_row, col1_w, card_h_left, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.22, shadow=True, zorder=3)
    draw_icon_preprocessing_doc(ax, col1_x + col1_w / 2, y_top_row + 1.28, scale=0.92, zorder=6)
    ax.text(col1_x + col1_w / 2, y_top_row + 0.58, "Dual-Stream Input", ha="center", va="center",
            fontsize=15.0, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(col1_x + col1_w / 2, y_top_row + 0.25, "1.pdf Clean + Safe NLTK", ha="center", va="center",
            fontsize=11.6, color="#515154", zorder=7)

    draw_rounded_card(ax, col1_x, y_bot_row, col1_w, card_h_left, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.22, shadow=True, zorder=3)
    draw_icon_data_layers(ax, col1_x + col1_w / 2, y_bot_row + 1.28, scale=0.92, zorder=6)
    ax.text(col1_x + col1_w / 2, y_bot_row + 0.58, "20-Feature & EBR", ha="center", va="center",
            fontsize=15.0, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(col1_x + col1_w / 2, y_bot_row + 0.25, "Parquet Store • λ=0.03", ha="center", va="center",
            fontsize=11.6, color="#515154", zorder=7)

    ax.plot([col1_x + col1_w / 2, col1_x + col1_w / 2], [y_bot_row + card_h_left, y_top_row], color="#86868B", lw=2.0, zorder=2)

    col2_x = 4.35
    col2_w = 3.35
    col3_x = 8.15
    col3_w = 3.35

    draw_rounded_card(ax, col2_x, y_top_row, col2_w, card_h_left, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.22, shadow=True, zorder=3)
    draw_icon_neural_network(ax, col2_x + col2_w / 2, y_top_row + 1.26, scale=0.92, zorder=6)
    ax.text(col2_x + col2_w / 2, y_top_row + 0.58, "BERT Backbone", ha="center", va="center",
            fontsize=15.0, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(col2_x + col2_w / 2, y_top_row + 0.25, "bert-base-uncased (768d)", ha="center", va="center",
            fontsize=11.6, color="#515154", zorder=7)

    draw_rounded_card(ax, col2_x, y_bot_row, col2_w, card_h_left, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.22, shadow=True, zorder=3)
    draw_icon_circular_arrow(ax, col2_x + col2_w / 2, y_bot_row + 1.28, scale=0.92, zorder=6)
    ax.text(col2_x + col2_w / 2, y_bot_row + 0.58, "Feature Fusion", ha="center", va="center",
            fontsize=15.0, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(col2_x + col2_w / 2, y_bot_row + 0.25, "Residual Gate (20 → 768d)", ha="center", va="center",
            fontsize=11.6, color="#515154", zorder=7)

    draw_rounded_card(ax, col3_x, y_top_row, col3_w, card_h_left, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.22, shadow=True, zorder=3)
    draw_icon_optimization(ax, col3_x + col3_w / 2, y_top_row + 1.28, scale=0.92, zorder=6)
    ax.text(col3_x + col3_w / 2, y_top_row + 0.58, "Rationale Masking", ha="center", va="center",
            fontsize=15.0, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(col3_x + col3_w / 2, y_top_row + 0.25, "Soft Mask m_a • Overlap Loss", ha="center", va="center",
            fontsize=11.6, color="#515154", zorder=7)

    draw_rounded_card(ax, col3_x, y_bot_row, col3_w, card_h_left, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.22, shadow=True, zorder=3)
    draw_icon_chart_nodes(ax, col3_x + col3_w / 2, y_bot_row + 1.28, scale=0.92, zorder=6)
    ax.text(col3_x + col3_w / 2, y_bot_row + 0.58, "CORN Ordinal Head", ha="center", va="center",
            fontsize=15.0, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(col3_x + col3_w / 2, y_bot_row + 0.25, "Rank-Consistent 1–5★ Rating", ha="center", va="center",
            fontsize=11.6, color="#515154", zorder=7)

    for y_mid in [y_top_row + card_h_left / 2, y_bot_row + card_h_left / 2]:
        ax.plot([col1_x + col1_w, col2_x], [y_mid, y_mid], color="#86868B", lw=2.0, zorder=2)
        ax.plot([col2_x + col2_w, col3_x], [y_mid, y_mid], color="#86868B", lw=2.0, zorder=2)

    ax.plot([col2_x + col2_w / 2, col2_x + col2_w / 2], [y_bot_row + card_h_left, y_top_row], color="#86868B", lw=2.0, zorder=2)
    ax.plot([col3_x + col3_w / 2, col3_x + col3_w / 2], [y_bot_row + card_h_left, y_top_row], color="#86868B", lw=2.0, zorder=2)

    right_box_x = 12.00
    right_box_y = 1.42
    right_box_w = 3.95
    right_box_h = 4.18

    for y_mid in [y_top_row + card_h_left / 2, y_bot_row + card_h_left / 2]:
        ax.plot([col3_x + col3_w, right_box_x - 0.04], [y_mid, y_mid], color="#86868B", lw=2.0, zorder=2)
        ax.plot([right_box_x - 0.16, right_box_x - 0.03, right_box_x - 0.16],
                [y_mid + 0.09, y_mid, y_mid - 0.09], color="#86868B", lw=2.0, solid_capstyle="round", zorder=2)

    draw_rounded_card(ax, right_box_x, right_box_y, right_box_w, right_box_h,
                      bg="#F5F5F7", edgecolor="#FFFFFF", lw=2.2, radius=0.26, shadow=True, zorder=3)
    rb_border = FancyBboxPatch(
        (right_box_x, right_box_y), right_box_w, right_box_h,
        boxstyle="round,pad=0.0,rounding_size=0.26",
        facecolor="none", edgecolor="#D8D8DE", linewidth=1.2, zorder=4
    )
    ax.add_patch(rb_border)

    rcx = right_box_x + right_box_w / 2
    ax.text(rcx, right_box_y + right_box_h - 0.36, "6-Aspect Adapters", ha="center", va="center",
            fontsize=15.8, fontweight="bold", color="#1D1D1F", zorder=7)

    aspect_names = ["Service", "Facility", "Exper.", "Amenity", "Loyalty", "Brand."]
    bar_w = 0.45
    bar_h = 1.42
    bar_gap = 0.11
    total_bars_w = 6 * bar_w + 5 * bar_gap
    bar_start_x = rcx - total_bars_w / 2
    bar_y = right_box_y + 2.12
    bar_centers = []

    for a_i, a_lbl in enumerate(aspect_names):
        bx = bar_start_x + a_i * (bar_w + bar_gap)
        bcx = bx + bar_w / 2
        bar_centers.append(bcx)
        pill = FancyBboxPatch(
            (bx, bar_y), bar_w, bar_h,
            boxstyle="round,pad=0.0,rounding_size=0.08",
            facecolor="#0099C6", edgecolor="none", zorder=5
        )
        ax.add_patch(pill)
        for d_i in range(3):
            ax.add_patch(Circle((bcx, bar_y + bar_h - 0.13 - d_i * 0.07), 0.017, facecolor="#FFFFFF", edgecolor="none", zorder=6))
        ax.text(bcx, bar_y + 0.52, a_lbl, ha="center", va="center", rotation=90,
                fontsize=9.6, fontweight="bold", color="#FFFFFF", zorder=7)

    y_tree_top = bar_y
    y_tree_horiz = bar_y - 0.18
    y_tree_bot = bar_y - 0.42
    for bcx in bar_centers:
        ax.plot([bcx, bcx], [y_tree_top, y_tree_horiz], color="#0099C6", lw=1.9, zorder=5)
    ax.plot([bar_centers[0], bar_centers[-1]], [y_tree_horiz, y_tree_horiz], color="#0099C6", lw=1.9, zorder=5)
    ax.plot([rcx, rcx], [y_tree_horiz, y_tree_bot], color="#0099C6", lw=1.9, zorder=5)

    out_card_w = right_box_w - 0.44
    out_card_h = 1.42
    out_card_x = rcx - out_card_w / 2
    out_card_y = right_box_y + 0.24
    draw_rounded_card(ax, out_card_x, out_card_y, out_card_w, out_card_h,
                      bg="#FFFFFF", edgecolor="#DCE6F2", lw=1.3, radius=0.18, shadow=False, zorder=5)
    ax.text(rcx, out_card_y + 1.08, "CRAM-ABSA Output", ha="center", va="center",
            fontsize=15.0, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(rcx, out_card_y + 0.68, "• Aspect: Mentioned List", ha="center", va="center",
            fontsize=11.8, fontweight="bold", color="#0066CC", zorder=7)
    ax.text(rcx, out_card_y + 0.30, "• Service..Branding: ★ + Text", ha="center", va="center",
            fontsize=11.6, fontweight="normal", color="#333336", zorder=7)

    pill_x = col1_x
    pill_w = (right_box_x + right_box_w) - col1_x
    pill_y = 0.42
    pill_h = 0.66
    draw_rounded_card(ax, pill_x, pill_y, pill_w, pill_h, bg="#FFFFFF", edgecolor="#ECECF0", lw=1.0, radius=0.32, shadow=False, zorder=3)
    pcx = pill_x + pill_w / 2
    pcy = pill_y + pill_h / 2
    split_x = pcx - 1.85
    ax.text(split_x - 0.08, pcy, "Multi-Task Joint Objective:",
            ha="right", va="center", fontsize=14.2, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(split_x + 0.08, pcy, "L_Total = L_6-Aspect + β1·L_CORN + β2·L_Overlap + β3·L_Rationale",
            ha="left", va="center", fontsize=14.2, fontweight="normal", color="#1D1D1F", zorder=7)

    for p in [
        "data/chart_img/412_abstract_architecture_of_proposed_method.png",
        "docs/main paper/chart_img/412_abstract_architecture_of_proposed_method.png",
    ]:
        plt.savefig(p, bbox_inches="tight", pad_inches=0.04, facecolor="none")
    plt.close(fig)
    print("Generated refined 412_abstract_architecture_of_proposed_method.png successfully.")

build_figure_412()


Generated refined 412_abstract_architecture_of_proposed_method.png successfully.


## 5. Feature Engineering


### 5.1 Tạo 20 Features


In [51]:
import math
import re
from collections import defaultdict, Counter

try:
    import nltk
    from nltk.corpus import stopwords
    try:
        NLTK_STOPWORDS = set(stopwords.words("english"))
    except LookupError:
        nltk.download("stopwords", quiet=True)
        nltk.download("punkt", quiet=True)
        nltk.download("averaged_perceptron_tagger", quiet=True)
        NLTK_STOPWORDS = set(stopwords.words("english"))
except Exception:
    NLTK_STOPWORDS = {
        "a", "an", "the", "and", "or", "if", "in", "on", "at", "to", "from", "by", "for",
        "with", "about", "as", "into", "like", "through", "after", "over", "between", "out",
        "of", "during", "before", "under", "around", "among", "is", "are", "was", "were",
        "be", "been", "being", "have", "has", "had", "do", "does", "did", "will", "would",
        "shall", "should", "can", "could", "may", "might", "must", "i", "me", "my", "myself",
        "we", "our", "ours", "ourselves", "you", "your", "yours", "he", "him", "his", "she",
        "her", "hers", "it", "its", "they", "them", "their", "theirs", "this", "that", "these",
        "those", "am", "so", "than", "too", "very", "just", "where", "when", "who", "whom",
        "which", "what", "there", "here", "all", "any", "both", "each", "few", "more", "most",
        "other", "some", "such", "only", "own", "same", "then", "once"
    }

CONTRACTIONS_MAP = {
    "can't": "cannot", "won't": "will not", "don't": "do not", "doesn't": "does not",
    "didn't": "did not", "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "couldn't": "could not", "wouldn't": "would not", "shouldn't": "should not",
    "n't": " not", "'re": " are", "'ve": " have", "'ll": " will", "'m": " am"
}

NEGATION_AND_CONTRAST_WORDS = {
    "not", "no", "never", "neither", "nor", "none", "nothing", "nowhere", "hardly",
    "barely", "scarcely", "without", "but", "however", "although", "though", "yet",
    "despite", "unfortunately", "except", "fault"
}

SAFE_STOPWORDS = NLTK_STOPWORDS - NEGATION_AND_CONTRAST_WORDS

OPINION_LEXICON_POS = {
    "good", "great", "excellent", "amazing", "wonderful", "friendly", "helpful", "clean",
    "spacious", "comfortable", "delicious", "lovely", "perfect", "best", "nice", "fantastic",
    "superb", "polite", "attentive", "convenient", "beautiful", "cozy", "relaxing", "awesome",
    "recommend", "recommended", "happy", "pleased", "impressed", "welcoming", "professional",
    "quiet", "modern", "fresh", "tasty", "worth", "luxury", "luxurious", "outstanding", "warm"
}

OPINION_LEXICON_NEG = {
    "bad", "poor", "terrible", "horrible", "awful", "slow", "rude", "dirty", "noisy", "small",
    "old", "outdated", "broken", "cold", "hard", "uncomfortable", "overpriced", "expensive",
    "worst", "disappointed", "disappointing", "unfriendly", "unhelpful", "smell", "smelly",
    "stain", "stained", "cockroach", "bug", "mold", "issue", "problem", "complaint", "wait",
    "waiting", "delayed", "mediocre", "basic", "boring", "crowded", "weak", "fault"
}

INTENSIFIERS = {
    "very", "extremely", "really", "super", "highly", "totally", "absolutely", "completely",
    "quite", "pretty", "so", "too", "exceptionally", "incredibly", "amazingly", "remarkably"
}

CANONICAL_ASPECTS = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]
ASPECT_ID_MAP = {asp: idx + 1 for idx, asp in enumerate(CANONICAL_ASPECTS)}
ASPECT_ID_MAP["None"] = 0

ASPECT_KEYWORDS = {
    "Service": {"staff", "service", "reception", "front", "desk", "manager", "waiter", "waitress", "concierge", "bellboy", "helpful", "friendly", "polite", "check", "checkin", "checkout", "attendant", "housekeeping", "host", "crew"},
    "Facility": {"room", "bed", "bathroom", "shower", "toilet", "air", "ac", "conditioning", "clean", "cleanliness", "noise", "noisy", "soundproof", "view", "balcony", "floor", "elevator", "lift", "pillow", "mattress", "furniture", "decor", "water", "carpet"},
    "Experience": {"stay", "stayed", "experience", "holiday", "vacation", "trip", "night", "overall", "atmosphere", "vibe", "relax", "relaxing", "enjoy", "enjoyed", "memorable", "time", "comfort", "comfortable", "quiet", "peaceful", "feeling"},
    "Amenity": {"breakfast", "buffet", "pool", "swimming", "spa", "gym", "fitness", "wifi", "internet", "restaurant", "bar", "food", "drink", "coffee", "tea", "lounge", "shuttle", "parking", "massage", "rooftop"},
    "Loyalty": {"back", "return", "again", "recommend", "recommended", "definitely", "choose", "future", "next", "visit", "repeat", "choice", "staying"},
    "Branding": {"star", "stars", "luxury", "brand", "hotel", "resort", "standard", "reputation", "class", "five", "four", "chain", "management", "value", "price", "worth", "quality", "prestigious", "accor", "sofitel", "pullman", "novotel", "marriott", "hilton", "intercontinental", "sheraton"}
}

def nltk_negation_aware_preprocess(text: str):

    lower_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        lower_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, lower_txt)
        lower_txt = lower_txt.replace(k, v)
    lower_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", lower_txt)
    raw_tokens = re.findall(r"[a-z]+", lower_txt)
    clean_tokens = [t for t in raw_tokens if t not in SAFE_STOPWORDS and len(t) > 1]
    return raw_tokens, clean_tokens

def preprocess_text_paper1(text: str):

    norm_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        norm_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, norm_txt)
        norm_txt = norm_txt.replace(k, v)
    norm_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", norm_txt)
    clean_review_text = re.sub(r"[^a-z0-9.,!?'\s-]", " ", norm_txt)
    clean_review_text = re.sub(r"\s+", " ", clean_review_text).strip()
    _, clean_tokens = nltk_negation_aware_preprocess(norm_txt)
    paper1_nltk_text = " ".join(clean_tokens)
    return clean_review_text, paper1_nltk_text

def compute_ebr_prototype_similarity(clean_tokens):

    if not clean_tokens:
        return 0.0
    tok_counts = Counter(clean_tokens)
    aspect_sims = []
    for asp in CANONICAL_ASPECTS:
        kw_set = ASPECT_KEYWORDS[asp]
        overlap_w = sum(cnt for tok, cnt in tok_counts.items() if tok in kw_set)
        norm_sim = overlap_w / math.sqrt(len(clean_tokens) * len(kw_set)) if len(clean_tokens) > 0 else 0.0
        aspect_sims.append(min(1.0, norm_sim * 3.5))
    top_sims = sorted(aspect_sims, reverse=True)[:3]
    return float(sum(top_sims) / len(top_sims)) if top_sims else 0.0

def infer_aspect_sentiment_from_text_nltk(raw_text: str, target_asp: str) -> float:

    sentences = [s.strip() for s in re.split(r"[.!?]+", str(raw_text)) if s.strip()]
    kw_set = ASPECT_KEYWORDS[target_asp]
    scores = []
    for sent in sentences:
        raw_toks, clean_toks = nltk_negation_aware_preprocess(sent)
        if any(t in kw_set for t in clean_toks):
            pos_hits = sum(1 for t in raw_toks if t in OPINION_LEXICON_POS)
            neg_hits = sum(1 for t in raw_toks if t in OPINION_LEXICON_NEG)

            for idx_t, tok in enumerate(raw_toks[:-1]):
                if tok in {"not", "never", "no", "hardly"}:
                    nxt = raw_toks[idx_t + 1]
                    if nxt in OPINION_LEXICON_POS:
                        pos_hits = max(0, pos_hits - 1)
                        neg_hits += 1
                    elif nxt in OPINION_LEXICON_NEG:
                        neg_hits = max(0, neg_hits - 1)
                        pos_hits += 1
            if pos_hits + neg_hits > 0:
                scores.append((pos_hits - neg_hits) / (pos_hits + neg_hits))
            else:
                scores.append(0.0)
    return float(round(sum(scores) / len(scores), 4)) if scores else 0.0

rev_time_map = df_reviews.set_index("review_id")[["hotel_name", "year", "month"]].copy()
rev_time_map["m_val"] = rev_time_map["year"].fillna(0).astype(int) * 12 + rev_time_map["month"].fillna(0).astype(int)

df_ann_time = df_annotations.join(rev_time_map, on="review_id")
max_time_per_hotel_aspect = df_ann_time.groupby(["hotel_name", "aspect"])["m_val"].max().to_dict()
max_time_per_hotel = df_reviews.groupby("hotel_name").apply(
    lambda g: (g["year"].fillna(0).astype(int) * 12 + g["month"].fillna(0).astype(int)).max(),
    include_groups=False
).to_dict()

lambda_decay = 0.03
sentiment_val_map = {"Positive": 1.0, "Neutral": 0.0, "Negative": -1.0}

features_dict = {}
grouped = df_annotations.groupby("review_id")

for _, row in df_reviews.iterrows():
    rid = int(row["review_id"])
    raw_text = str(row["review_text"])
    char_len = int(row["char_length"])
    word_cnt = int(row["word_count"])
    h_name = str(row["hotel_name"]).strip()
    rev_m = int(row["year"]) * 12 + int(row["month"])
    hotel_max_m = max_time_per_hotel.get(h_name, rev_m)
    delta_h_months = max(0, hotel_max_m - rev_m)
    recency_w = float(math.exp(-lambda_decay * delta_h_months))

    raw_tokens, clean_tokens = nltk_negation_aware_preprocess(raw_text)
    total_raw_toks = max(len(raw_tokens), 1)
    clean_token_ratio = float(min(len(clean_tokens) / total_raw_toks, 1.0))

    opinion_neg_cnt = sum(
        1 for t in raw_tokens
        if (t in OPINION_LEXICON_POS or t in OPINION_LEXICON_NEG or t in NEGATION_AND_CONTRAST_WORDS or t in INTENSIFIERS)
    )
    opinion_neg_density = float(min(opinion_neg_cnt / total_raw_toks, 1.0))
    ebr_proto_sim = compute_ebr_prototype_similarity(clean_tokens)

    if rid in grouped.groups:
        sub_ann = grouped.get_group(rid)

        asp_count = int(sub_ann["aspect"].nunique())

        asp_mode = sub_ann["aspect"].mode()
        dominant_aspect_str = str(asp_mode.iloc[0]) if not asp_mode.empty else "None"
        dominant_aspect = int(ASPECT_ID_MAP.get(dominant_aspect_str, 0))

        pos_cnt = int((sub_ann["sentiment"] == "Positive").sum())
        neg_cnt = int((sub_ann["sentiment"] == "Negative").sum())
        tot_cnt = int(len(sub_ann))
        net_sentiment = float((pos_cnt - neg_cnt) / tot_cnt) if tot_cnt > 0 else 0.0

        has_pos = pos_cnt > 0
        has_neg = neg_cnt > 0
        sent_conflict = 1 if (has_pos and has_neg) else 0

        total_span_chars = int((sub_ann["end_char"] - sub_ann["start_char"]).sum())
        evidence_density = float(min(total_span_chars / char_len, 1.0)) if char_len > 0 else 0.0

        avg_rat_len = float((sub_ann["end_char"] - sub_ann["start_char"]).mean()) if tot_cnt > 0 else 0.0

        has_overlap = 0
        records = sub_ann[["aspect", "sentiment", "start_char", "end_char"]].to_dict("records")
        n_spans = len(records)
        for i in range(n_spans):
            for j in range(i + 1, n_spans):
                if records[i]["aspect"] != records[j]["aspect"]:
                    if not (records[i]["end_char"] <= records[j]["start_char"] or records[i]["start_char"] >= records[j]["end_char"]):
                        has_overlap = 1
                        break
            if has_overlap == 1:
                break

        def get_aspect_score(target_asp):
            asp_recs = [r for r in records if r["aspect"] == target_asp]
            if not asp_recs:
                return 0.0
            weighted_sum = 0.0
            total_len = 0.0
            for r in asp_recs:
                slen = max(1, int(r["end_char"]) - int(r["start_char"]))
                sval = sentiment_val_map.get(r["sentiment"], 0.0)
                weighted_sum += slen * sval
                total_len += slen
            return float(round(weighted_sum / total_len, 4)) if total_len > 0 else 0.0

        serv_score = get_aspect_score("Service")
        fac_score = get_aspect_score("Facility")
        exp_score = get_aspect_score("Experience")
        amen_score = get_aspect_score("Amenity")
        loy_score = get_aspect_score("Loyalty")
        brand_score = get_aspect_score("Branding")

        coverage_ratio = float(min(asp_count / 6.0, 1.0))
        consensus_strength = float(abs(net_sentiment))

        asp_dict = defaultdict(list)
        for r in records:
            asp_dict[r["aspect"]].append(sentiment_val_map.get(r["sentiment"], 0.0))

        asp_temp_scores = []
        for asp, s_vals in asp_dict.items():
            max_m = max_time_per_hotel_aspect.get((h_name, asp), rev_m)
            delta_months = max(0, max_m - rev_m)
            w_time = math.exp(-lambda_decay * delta_months)
            avg_s = sum(s_vals) / len(s_vals)
            asp_temp_scores.append(w_time * avg_s)

        temporal_aspect_sentiment = float(sum(asp_temp_scores) / len(asp_temp_scores)) if len(asp_temp_scores) > 0 else 0.0

    else:

        aspect_scores_inferred = {asp: infer_aspect_sentiment_from_text_nltk(raw_text, asp) for asp in CANONICAL_ASPECTS}
        active_aspects = [asp for asp, sc in aspect_scores_inferred.items() if abs(sc) > 0 or any(t in ASPECT_KEYWORDS[asp] for t in clean_tokens)]
        asp_count = len(active_aspects)
        dominant_aspect = ASPECT_ID_MAP.get(active_aspects[0], 0) if active_aspects else 0
        active_vals = [aspect_scores_inferred[a] for a in active_aspects] if active_aspects else [0.0]
        net_sentiment = float(sum(active_vals) / len(active_vals))
        sent_conflict = 1 if (any(v > 0 for v in active_vals) and any(v < 0 for v in active_vals)) else 0
        evidence_density = clean_token_ratio
        avg_rat_len = float(char_len / max(asp_count, 1))
        has_overlap = 0
        serv_score = aspect_scores_inferred["Service"]
        fac_score = aspect_scores_inferred["Facility"]
        exp_score = aspect_scores_inferred["Experience"]
        amen_score = aspect_scores_inferred["Amenity"]
        loy_score = aspect_scores_inferred["Loyalty"]
        brand_score = aspect_scores_inferred["Branding"]
        coverage_ratio = float(min(asp_count / 6.0, 1.0))
        consensus_strength = float(abs(net_sentiment))
        temporal_aspect_sentiment = float(recency_w * net_sentiment)

    features_dict[rid] = {
        "review_id": rid,
        "feat_aspect_count": int(asp_count),
        "feat_dominant_aspect": int(dominant_aspect),
        "feat_net_sentiment_score": float(round(net_sentiment, 4)),
        "feat_sentiment_conflict": int(sent_conflict),
        "feat_evidence_density": float(round(evidence_density, 4)),
        "feat_avg_rationale_len": float(round(avg_rat_len, 2)),
        "feat_has_cross_overlap": int(has_overlap),
        "feat_service_sentiment": float(round(serv_score, 4)),
        "feat_facility_sentiment": float(round(fac_score, 4)),
        "feat_experience_sentiment": float(round(exp_score, 4)),
        "feat_amenity_sentiment": float(round(amen_score, 4)),
        "feat_loyalty_sentiment": float(round(loy_score, 4)),
        "feat_branding_sentiment": float(round(brand_score, 4)),
        "feat_aspect_coverage_ratio": float(round(coverage_ratio, 4)),
        "feat_recency_weight": float(round(recency_w, 4)),
        "feat_consensus_strength": float(round(consensus_strength, 4)),
        "feat_temporal_aspect_sentiment": float(round(temporal_aspect_sentiment, 4)),
        "feat_nltk_clean_token_ratio": float(round(clean_token_ratio, 4)),
        "feat_nltk_opinion_negation_density": float(round(opinion_neg_density, 4)),
        "feat_ebr_aspect_prototype_sim": float(round(ebr_proto_sim, 4))
    }

FEATURE_COLS_20 = [c for c in list(features_dict[1].keys()) if c != "review_id"]
print(f"Đã trích xuất thành công {len(FEATURE_COLS_20)} features cho {len(features_dict):,} bài đánh giá:")
print("Danh sách 20 features:", FEATURE_COLS_20)

cleaned_text_pairs = [preprocess_text_paper1(t) for t in df_reviews["review_text"]]
df_reviews["clean_review_text"] = [p[0] for p in cleaned_text_pairs]
df_reviews["paper1_nltk_text"] = [p[1] for p in cleaned_text_pairs]
df_features = pd.DataFrame(list(features_dict.values()))
df_merged = pd.merge(df_reviews, df_features, on="review_id")
display(df_features.head())


Đã trích xuất thành công 20 features cho 9,990 bài đánh giá:
Danh sách 20 features: ['feat_aspect_count', 'feat_dominant_aspect', 'feat_net_sentiment_score', 'feat_sentiment_conflict', 'feat_evidence_density', 'feat_avg_rationale_len', 'feat_has_cross_overlap', 'feat_service_sentiment', 'feat_facility_sentiment', 'feat_experience_sentiment', 'feat_amenity_sentiment', 'feat_loyalty_sentiment', 'feat_branding_sentiment', 'feat_aspect_coverage_ratio', 'feat_recency_weight', 'feat_consensus_strength', 'feat_temporal_aspect_sentiment', 'feat_nltk_clean_token_ratio', 'feat_nltk_opinion_negation_density', 'feat_ebr_aspect_prototype_sim']


,review_id,feat_aspect_count,feat_dominant_aspect,feat_net_sentiment_score,feat_sentiment_conflict,feat_evidence_density,feat_avg_rationale_len,feat_has_cross_overlap,feat_service_sentiment,feat_facility_sentiment,...,feat_amenity_sentiment,feat_loyalty_sentiment,feat_branding_sentiment,feat_aspect_coverage_ratio,feat_recency_weight,feat_consensus_strength,feat_temporal_aspect_sentiment,feat_nltk_clean_token_ratio,feat_nltk_opinion_negation_density,feat_ebr_aspect_prototype_sim
0,1,3,2,0.6,1,0.5572,38.00,0,-0.6,1.0,...,0.0,0.0,0.0,0.5000,0.0672,0.6,0.0968,0.4328,0.1791,0.4607
1,2,3,3,1.0,0,0.6107,68.57,0,1.0,1.0,...,0.0,0.0,0.0,0.5000,0.0758,1.0,0.0790,0.6462,0.0923,0.3807
2,3,4,3,1.0,0,0.5668,40.00,0,1.0,1.0,...,0.0,1.0,0.0,0.6667,0.9418,1.0,0.9709,0.5595,0.1429,0.2338
3,4,3,1,1.0,0,0.7238,65.50,0,1.0,1.0,...,0.0,1.0,0.0,0.5000,0.9704,1.0,0.9704,0.5052,0.0619,0.3262
4,5,2,2,1.0,0,0.8471,43.20,0,1.0,1.0,...,0.0,0.0,0.0,0.3333,0.6977,1.0,0.6977,0.6154,0.1538,0.3102


### 5.2 Lưu trữ DataFrame thành tệp Parquet và xuất CSDL


In [52]:
os.makedirs("data/df", exist_ok=True)
os.makedirs("data/feature", exist_ok=True)

ann_grouped = df_annotations.groupby("review_id").apply(
    lambda g: g[["annotation_id", "aspect", "sentiment", "start_char", "end_char", "span_text"]].to_dict("records"),
    include_groups=False
).reset_index(name="aspect_annotations")

df_master = pd.merge(df_merged, ann_grouped, on="review_id", how="left")
df_master["aspect_annotations"] = df_master["aspect_annotations"].apply(lambda x: x if isinstance(x, list) else [])

parquet_path = "data/df/hotel_reviews.parquet"
df_master.to_parquet(parquet_path, compression="snappy", index=False)
print(f"Đã xuất tệp Parquet duy nhất của dự án tại: '{parquet_path}' ({len(df_master):,} dòng, {len(df_master.columns)} cột, {len(df_annotations):,} annotations lồng nhau).")

db_path = "data/df/chatbot.db"
df_features.to_sql("review_features", conn, if_exists="replace", index=False)
cursor.execute("CREATE INDEX IF NOT EXISTS idx_feat_review_id ON review_features(review_id);")
conn.commit()
print(f"Đã lưu bảng 'review_features' ({len(df_features):,} dòng) vào SQLite tại '{db_path}'.")

feature_json_records = []
for idx, item in enumerate(raw_json, 1):
    item_copy = dict(item)
    feat_vals = features_dict.get(idx, {})
    item_copy["features"] = feat_vals
    feature_json_records.append(item_copy)

feature_json_path = "data/feature/TripAdvisor_Feature.json"
dump_bytes = orjson.dumps(
    feature_json_records,
    option=orjson.OPT_INDENT_2 | orjson.OPT_SERIALIZE_NUMPY
)

with open(feature_json_path, "wb") as f:
    f.write(dump_bytes)

print(f"Đã xuất '{feature_json_path}' thành công ({len(feature_json_records):,} bản ghi)")

conn.close()
print("Hoàn tất toàn bộ quy trình")


Đã xuất tệp Parquet duy nhất của dự án tại: 'data/df/hotel_reviews.parquet' (9,990 dòng, 39 cột, 54,326 annotations lồng nhau).
Đã lưu bảng 'review_features' (9,990 dòng) vào SQLite tại 'data/df/chatbot.db'.
Đã xuất 'data/feature/TripAdvisor_Feature.json' thành công (9,990 bản ghi)
Hoàn tất toàn bộ quy trình


### 5.3 Tổng hợp Đa Đánh giá theo Trọng số Thời gian


In [53]:
import math
import numpy as np
import pandas as pd

def demonstrate_multi_review_aspect_synthesis(df_master, hotel_target="Khách sạn pullman hà nội", aspect_target="Amenity", lambda_decay=0.03, top_k=5):

    hotel_revs = df_master[df_master["hotel_name"].str.lower() == hotel_target.lower()].copy()
    if hotel_revs.empty:
        hotel_revs = df_master[df_master["hotel_name"].str.lower().str.contains("pullman", na=False)].copy()

    if hotel_revs.empty:
        print(f"Không tìm thấy review cho khách sạn: {hotel_target}")
        return

    actual_hotel_name = hotel_revs.iloc[0]["hotel_name"]
    hotel_revs["year_int"] = pd.to_numeric(hotel_revs["year"], errors="coerce").fillna(2015).astype(int)
    hotel_revs["month_int"] = pd.to_numeric(hotel_revs["month"], errors="coerce").fillna(1).astype(int)
    hotel_revs["m_val"] = hotel_revs["year_int"] * 12 + hotel_revs["month_int"]
    max_m = hotel_revs["m_val"].max()

    hotel_revs["delta_months"] = np.maximum(0, max_m - hotel_revs["m_val"])
    hotel_revs["recency_weight"] = np.exp(-lambda_decay * hotel_revs["delta_months"])

    def mentions_aspect(ann_list):
        if not hasattr(ann_list, "__iter__"): return False
        return any(a.get("aspect") == aspect_target for a in ann_list if isinstance(a, dict))

    def get_aspect_sentiment(ann_list):
        if not hasattr(ann_list, "__iter__"): return 0
        sents = [a.get("sentiment") for a in ann_list if isinstance(a, dict) and a.get("aspect") == aspect_target]
        if not sents: return 0
        pos = sents.count("Positive")
        neg = sents.count("Negative")
        return 1 if pos > neg else (-1 if neg > pos else 0)

    def get_aspect_spans(ann_list):
        if not hasattr(ann_list, "__iter__"): return []
        spans = []
        for a in ann_list:
            if isinstance(a, dict) and a.get("aspect") == aspect_target and a.get("span_text"):
                spans.append(a.get("span_text"))
        return spans

    hotel_revs["has_aspect"] = hotel_revs["aspect_annotations"].apply(mentions_aspect)
    aspect_subset = hotel_revs[hotel_revs["has_aspect"]].copy()
    if aspect_subset.empty:
        aspect_subset = hotel_revs.copy()

    aspect_subset["aspect_sent_val"] = aspect_subset["aspect_annotations"].apply(get_aspect_sentiment)
    aspect_subset["evidence_spans"] = aspect_subset["aspect_annotations"].apply(get_aspect_spans)

    top_candidates = aspect_subset.sort_values(by="recency_weight", ascending=False).head(top_k)

    simple_avg_sent = top_candidates["aspect_sent_val"].mean()
    weighted_sent = np.average(top_candidates["aspect_sent_val"], weights=top_candidates["recency_weight"])

    pos_count = (top_candidates["aspect_sent_val"] > 0).sum()
    pos_pct = int(round(pos_count / len(top_candidates) * 100)) if len(top_candidates) > 0 else 0

    all_spans = []
    for spans in top_candidates["evidence_spans"]:
        all_spans.extend(spans)

    print("=" * 85)
    print(f"THỰC NGHIỆM TRÍCH XUẤT ĐA REVIEW VÀ TỔNG HỢP THEO KHÍA CẠNH: {aspect_target}")
    print(f"Khách sạn: {actual_hotel_name} | Số lượng reviews khớp khía cạnh: {len(top_candidates)} / {len(aspect_subset)}")
    print("=" * 85)

    print("\nChi tiết các bằng chứng trích xuất từ đánh giá gốc:")
    for idx, (_, row) in enumerate(top_candidates.iterrows(), 1):
        date_str = f"{row['year_int']}-{row['month_int']:02d}"
        sent_str = "Positive" if row["aspect_sent_val"] > 0 else ("Negative" if row["aspect_sent_val"] < 0 else "Neutral")
        span_sample = row["evidence_spans"][0] if row["evidence_spans"] else str(row.get("review_text", ""))[:90]
        print(f"  [{idx}] Ngày: {date_str} (delta: {row['delta_months']} tháng) | Trọng số: {row['recency_weight']*100:.1f}% | Cảm xúc: {sent_str}")
        print(f"      Dẫn chứng trích đoạn: \"{span_sample[:95]}...\"")

    print("\nTổng hợp phân tích đa chiều:")
    print(f"  • Tỷ lệ tích cực: {pos_pct}%")
    print(f"  • Điểm cảm xúc có trọng số thời gian: {weighted_sent:+.3f}")

    print("\nVí dụ câu trả lời tự nhiên được tổng hợp từ dữ liệu trên (trả lời câu hỏi: This hotel have swimming pool?):")
    if any("pool" in str(s).lower() for s in all_spans):
        pool_spans = [s for s in all_spans if "pool" in str(s).lower()]
        ans = f"Có, {actual_hotel_name} được trang bị hồ bơi tiện nghi. Nhiều du khách ghi nhận \"{pool_spans[0]}\", với chất lượng dịch vụ duy trì rất tốt theo các đánh giá gần đây."
    else:
        ans = f"Về khía cạnh {aspect_target} tại {actual_hotel_name}, các tiện ích như ẩm thực, bữa sáng và khu vực thư giãn được đánh giá rất cao ({pos_pct}% tích cực), trong đó các bài đánh giá gần đây đặc biệt khen ngợi chất lượng không gian và trải nghiệm tiện nghi."
    print(f"  \"{ans}\"")
    print("=" * 85)

demonstrate_multi_review_aspect_synthesis(df_master, hotel_target="Khách sạn pullman hà nội", aspect_target="Amenity")


THỰC NGHIỆM TRÍCH XUẤT ĐA REVIEW VÀ TỔNG HỢP THEO KHÍA CẠNH: Amenity
Khách sạn: Khách sạn pullman hà nội | Số lượng reviews khớp khía cạnh: 5 / 6

Chi tiết các bằng chứng trích xuất từ đánh giá gốc:
  [1] Ngày: 2023-03 (delta: 0 tháng) | Trọng số: 100.0% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: "Club Lounge excellent, good variety of hot and cold food, staff attentive..."
  [2] Ngày: 2018-11 (delta: 52 tháng) | Trọng số: 21.0% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: "Taxis readily available on call and that helps your commute..."
  [3] Ngày: 2017-08 (delta: 67 tháng) | Trọng số: 13.4% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: "surrounding area is full of interior materials and motorcycle shops. nearest convenience store ..."
  [4] Ngày: 2016-12 (delta: 75 tháng) | Trọng số: 10.5% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: "Checking in was easy..."
  [5] Ngày: 2016-03 (delta: 84 tháng) | Trọng số: 8.0% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: "One ca

### 5.4 Tiền xử lý Văn bản và Phân chia Tập dữ liệu Huấn luyện


In [54]:
import re
import sqlite3
import unicodedata
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

print("Bắt đầu tiền xử lý văn bản theo phương pháp paper 1.pdf và đồng bộ tệp hotel_reviews.parquet...")

parquet_path = "data/df/hotel_reviews.parquet"
df_proc = pd.read_parquet(parquet_path)

cleaned_pairs = [preprocess_text_paper1(txt) for txt in df_proc["review_text"]]
df_proc["clean_review_text"] = [p[0] for p in cleaned_pairs]
df_proc["paper1_nltk_text"] = [p[1] for p in cleaned_pairs]

def clean_annotation_spans(anns):
    if not isinstance(anns, (list, np.ndarray)):
        return []
    updated = []
    for ann in anns:
        if isinstance(ann, dict):
            item = dict(ann)
            c_span, nltk_span = preprocess_text_paper1(item.get("span_text", ""))
            item["clean_span_text"] = c_span
            item["paper1_span_tokens"] = nltk_span
            updated.append(item)
    return updated

df_proc["aspect_annotations"] = df_proc["aspect_annotations"].apply(clean_annotation_spans)

sample_paper1 = "I couldn't imagine a more beautiful place to stay. The staff were amazingly friendly. Could not fault the facilities and service. We didn't want to leave! We will be back for sure and won't stay anywhere else!"
sample_clean, sample_nltk = preprocess_text_paper1(sample_paper1)
print("Minh họa tiền xử lý văn bản theo paper 1.pdf (Table 4):")
print("  • Raw Text          :", sample_paper1)
print("  • clean_review_text :", sample_clean)
print("  • paper1_nltk_text  :", sample_nltk)

df_proc["ordinal_rating"] = df_proc["rating_5"].clip(1, 5).astype(int) - 1
for k in range(4):
    df_proc[f"corn_c{k}"] = (df_proc["ordinal_rating"] > k).astype(int)

def clean_hotel_name(raw_name: str) -> str:
    if not raw_name:
        return "Hotel"
    text = str(raw_name).replace("Đ", "D").replace("đ", "d")
    norm = unicodedata.normalize("NFKD", text)
    ascii_text = "".join(c for c in norm if not unicodedata.combining(c))
    clean_str = ascii_text.replace("\xa0", " ").strip()
    clean_str = re.sub(r"^(khach\s*san|khu\s*nghi\s*duong|ks|hotel|resort|the)\s+", "", clean_str, flags=re.IGNORECASE).strip()
    clean_str = re.sub(r"\s*\.*$", "", clean_str).strip()
    return clean_str.title() if clean_str else "Hotel"

df_proc["clean_hotel_name"] = df_proc["hotel_name"].apply(clean_hotel_name)

sss_train = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_indices, temp_indices = next(sss_train.split(df_proc, df_proc["rating_5"]))

df_temp = df_proc.iloc[temp_indices]
sss_val_test = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_sub_idx, test_sub_idx = next(sss_val_test.split(df_temp, df_proc.iloc[temp_indices]["rating_5"]))
val_indices = temp_indices[val_sub_idx]
test_indices = temp_indices[test_sub_idx]

df_proc["split"] = "train"
df_proc.loc[val_indices, "split"] = "val"
df_proc.loc[test_indices, "split"] = "test"

df_proc.to_parquet(parquet_path, compression="snappy", index=False)

db_path = "data/df/chatbot.db"
conn = sqlite3.connect(db_path)
df_proc[["review_id", "clean_review_text", "paper1_nltk_text", "split", "ordinal_rating", "corn_c0", "corn_c1", "corn_c2", "corn_c3"]].to_sql(
    "train_split_info", conn, if_exists="replace", index=False
)
cursor = conn.cursor()
cursor.execute("CREATE INDEX IF NOT EXISTS idx_split ON train_split_info(split);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ord_rating ON train_split_info(ordinal_rating);")
conn.commit()
conn.close()

split_counts = df_proc["split"].value_counts()
print(f"\nĐã lưu tệp Parquet hoàn chỉnh tại: '{parquet_path}' ({len(df_proc):,} dòng, {len(df_proc.columns)} cột - gồm cả clean_review_text & paper1_nltk_text)")
print("Thống kê phân chia tập dữ liệu:")
for s_name in ["train", "val", "test"]:
    cnt = split_counts.get(s_name, 0)
    pct = (cnt / len(df_proc)) * 100
    print(f"  • Tập {s_name.upper():<5}: {cnt:,} bài đánh giá ({pct:.1f}%)")

print("\nPhân bố tỷ lệ sao (% trên từng tập sau khi Stratified Split):")
dist_table = pd.crosstab(df_proc["rating_5"], df_proc["split"], normalize="columns") * 100
print(dist_table.round(2).to_string())

Bắt đầu tiền xử lý văn bản theo phương pháp paper 1.pdf và đồng bộ tệp hotel_reviews.parquet...
Minh họa tiền xử lý văn bản theo paper 1.pdf (Table 4):
  • Raw Text          : I couldn't imagine a more beautiful place to stay. The staff were amazingly friendly. Could not fault the facilities and service. We didn't want to leave! We will be back for sure and won't stay anywhere else!
  • clean_review_text : i could not imagine a more beautiful place to stay. the staff were amazingly friendly. could not fault the facilities and service. we did not want to leave! we will be back for sure and will not stay anywhere else!
  • paper1_nltk_text  : could not imagine beautiful place stay staff amazingly friendly could not fault facilities service not want leave back sure not stay anywhere else

Đã lưu tệp Parquet hoàn chỉnh tại: 'data/df/hotel_reviews.parquet' (9,990 dòng, 46 cột - gồm cả clean_review_text & paper1_nltk_text)
Thống kê phân chia tập dữ liệu:
  • Tập TRAIN: 7,992 bài đánh giá (80